In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [3]:
def analyze_review(review):
    system_prompt = """
    You are a product review sentiment analyst.
    Read the review provided by the user and classify its sentiment.

    Return:
    Sentiment = Positive, Negative, or Neutral
    Confidence Score = a value between 0.0 and 1.0
    Reason = a short explanation for the sentiment.

    Use this exact format:
    Sentiment: <Positive/Negative/Neutral>
    Confidence: <0.0 to 1.0>
    Reason: <Short explanation>
    """

    response = client.chat.completions.create(
        model=os.getenv("GROQ_MODEL", "openai/gpt-oss-20b"),
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": review}
        ],
        temperature=0
    )

    print(response.choices[0].message.content.strip())

In [4]:
reviews = [
    "This product is amazing and works perfectly!",
    "Terrible quality. I regret buying it.",
    "The product arrived yesterday."
]

analyze_review(reviews[0])
analyze_review(reviews[1])
analyze_review(reviews[2])

Sentiment: Positive
Confidence: 0.99
Reason: The review praises the product as amazing and fully functional.
Sentiment: Negative
Confidence: 0.95
Reason: The review explicitly states poor quality and regret, indicating strong dissatisfaction.
Sentiment: Neutral  
Confidence: 0.95  
Reason: The statement simply reports the arrival of the product without expressing any positive or negative feelings.


In [12]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

def analyze_review(review):
    system_prompt = """
    You are a product review sentiment analyst.
    Classify the review as positive, negative, or neutral.
    Return strictly valid JSON with three fields:
    sentiment, confidence (0.0 to 1.0), reason.
    Return JSON only.
    """

    response = client.chat.completions.create(
        model=os.getenv("GROQ_MODEL", "openai/gpt-oss-20b"),
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": review}
        ],
        temperature=0
    )

    raw_reply = response.choices[0].message.content.strip()

    try:
        return json.loads(raw_reply)
    except json.JSONDecodeError:
        return {"error": "Invalid JSON", "raw_reply": raw_reply}


def validate_sentiment_reply(raw_reply: str) -> bool:
    try:
        json.loads(raw_reply)
        return True
    except json.JSONDecodeError:
        return False


review1 = analyze_review("This product is excellent!")
print(review1)
print(type(review1))  # dict

{'sentiment': 'positive', 'confidence': 0.95, 'reason': 'The review contains the positive adjective "excellent," indicating a favorable opinion of the product.'}
<class 'dict'>


In [13]:
from pydantic import BaseModel, Field
from typing import Literal

class SentimentResult(BaseModel):
    """Sentiment results from the LLM."""

    sentiment: Literal["positive", "negative", "neutral"]
    confidence: float = Field(ge=0.0, le=1.0)
    reason: str


result = SentimentResult.model_validate(review1)
print(result)

sentiment='positive' confidence=0.95 reason='The review contains the positive adjective "excellent," indicating a favorable opinion of the product.'


In [14]:
def review_by_sentiment(result: SentimentResult) -> str:
    if result.sentiment == "positive":
        return "Positive review"
    elif result.sentiment == "negative":
        return "Negative review, call the customer"
    else:
        return "Neutral review, no action needed"


print(review_by_sentiment(result))

Positive review
